# 21.3 Knowledge representation and reasoning

A representation is useful when the facts you store make the conclusions you need cheap, explainable, and safe.

Knowledge representation connects raw logic to useful AI systems. Graphs, inheritance, defaults, and explicit explanation paths make symbolic conclusions inspectable.

Save a copy to Drive to edit.

In [ ]:

import random

import matplotlib.pyplot as plt

random.seed(2103)


def ancestor_paths(graph, node):
    paths = []
    stack = [(node, [node])]
    while stack:
        current, path = stack.pop()
        parents = graph.get(current, [])
        if not parents:
            paths.append(path)
        for parent in parents:
            stack.append((parent, path + [parent]))
    return paths


def inherit_properties(graph, properties, defaults, node):
    inherited = set()
    evidence = []
    scores = dict()
    for path in ancestor_paths(graph, node):
        for item in path:
            for prop in properties.get(item, set()):
                inherited.add(prop)
                evidence.append((item, prop, list(path)))
            for prop, score in defaults.get(item, {}).items():
                scores[prop] = scores.get(prop, 0) + score
                evidence.append((item, prop, list(path), score))
    blocked = {prop for prop, score in scores.items() if score <= 0}
    active_defaults = {prop for prop, score in scores.items() if score > 0}
    final_props = (inherited | active_defaults) - blocked
    return {"properties": final_props, "scores": scores, "blocked": blocked, "evidence": evidence}


def make_kr_ladder():
    return [
        {"name": "D1 Tweety", "graph": {"Tweety": ["Penguin"], "Penguin": ["Bird"], "Bird": []}, "properties": {"Bird": {"wings"}, "Penguin": {"swims"}}, "defaults": {"Bird": {"flies": 1}, "Penguin": {"flies": -1}}, "query": "Tweety", "expected": {"wings", "swims"}},
        {"name": "D2 animal taxonomy", "graph": {"Nemo": ["Clownfish"], "Clownfish": ["Fish"], "Fish": ["Animal"], "Animal": []}, "properties": {"Animal": {"alive"}, "Fish": {"gills"}, "Clownfish": {"stripes"}}, "defaults": {}, "query": "Nemo", "expected": {"alive", "gills", "stripes"}},
        {"name": "D3 conflicting defaults", "graph": {"RobotBird": ["Robot", "Bird"], "Robot": ["Artifact"], "Bird": ["Animal"], "Artifact": [], "Animal": []}, "properties": {"Artifact": {"manufactured"}, "Animal": {"observed"}}, "defaults": {"Bird": {"flies": 1}, "Robot": {"flies": -1}}, "query": "RobotBird", "expected": {"manufactured", "observed"}},
        {"name": "D4 product support", "graph": {"Case7": ["RefundCase"], "RefundCase": ["BillingCase"], "BillingCase": ["SupportCase"], "SupportCase": []}, "properties": {"SupportCase": {"ticket"}, "BillingCase": {"financial"}, "RefundCase": {"needs_receipt"}}, "defaults": {"BillingCase": {"auto_route": 1}}, "query": "Case7", "expected": {"ticket", "financial", "needs_receipt", "auto_route"}},
        {"name": "D5 enterprise policy", "graph": {"AdaAccess": ["ContractorAccess", "ResearchAccess"], "ContractorAccess": ["ExternalAccess"], "ResearchAccess": ["SensitiveAccess"], "ExternalAccess": ["Access"], "SensitiveAccess": ["Access"], "Access": []}, "properties": {"Access": {"logged"}, "SensitiveAccess": {"reviewed"}, "ExternalAccess": {"sponsored"}}, "defaults": {"Access": {"allow": 1}, "ExternalAccess": {"allow": -1}, "ResearchAccess": {"allow": 1}}, "query": "AdaAccess", "expected": {"logged", "reviewed", "sponsored", "allow"}},
    ]


## The concept, built once (D1)

The graph $Tweety	o Penguin	o Bird$ lets properties flow down the path. Set union gives $Props(Tweety)=\{wings,swims\}$, while default scores $+1-1=0$ block flying.

In [ ]:

graph = {"Tweety": ["Penguin"], "Penguin": ["Bird"], "Bird": []}
properties = {"Bird": {"wings"}, "Penguin": {"swims"}}
defaults = {"Bird": {"flies": 1}, "Penguin": {"flies": -1}}

d1 = inherit_properties(graph, properties, defaults, "Tweety")
print(d1["properties"])
print(d1["scores"])


The lesson asserts exactly $2$ inherited concrete properties and a blocked default with score $+1-1=0$. The explanation path, not just the label, is part of the answer.

In [ ]:

assert d1["properties"] == {"wings", "swims"}
assert len(d1["properties"]) == 2
assert d1["scores"]["flies"] == 0
assert "flies" in d1["blocked"]


## The dataset ladder

The ladder grows from Tweety to multi-inheritance enterprise policy data where defaults compete and explanations are required.

In [ ]:

kr_ladder = make_kr_ladder()

for rung in kr_ladder:
    nodes = sorted(rung["graph"].keys())
    print(rung["name"], "nodes", len(nodes), "query", rung["query"], "sample", nodes[:4])


## Run the same method across D1-D5

Every rung runs the same inheritance function. Correctness checks expected properties, and cost is the number of evidence steps in the explanation.

In [ ]:

kr_results = []
for rung in kr_ladder:
    result = inherit_properties(rung["graph"], rung["properties"], rung["defaults"], rung["query"])
    correct = rung["expected"].issubset(result["properties"])
    kr_results.append({
        "name": rung["name"],
        "nodes": len(rung["graph"]),
        "steps": len(result["evidence"]),
        "correct": correct,
        "result": result,
    })

for row in kr_results:
    print(row["name"], row["nodes"], row["steps"], row["correct"], row["result"]["properties"])


## Results visualization

The panels list semantic-network explanations. The curve compares graph size with explanation length.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
flat_axes = axes.ravel()

for ax, row in zip(flat_axes[:5], kr_results):
    lines = []
    for item in row["result"]["evidence"][:5]:
        lines.append(str(item))
    ax.text(0.03, 0.95, "\n".join(lines), va="top", fontsize=8)
    ax.set_title(row["name"])
    ax.axis("off")

ax = flat_axes[5]
ax.plot([row["nodes"] for row in kr_results], [row["steps"] for row in kr_results], marker="o")
ax.set_xlabel("graph nodes")
ax.set_ylabel("explanation steps")
ax.set_title("steps vs graph size")
plt.tight_layout()


## Pitfall on the hardest rung

Pitfall: choosing a representation that hides the constraint. If D5 stores only a final allow label, the override reasons disappear; the fix keeps defaults, overrides, and edges explicit.

In [ ]:

hard = kr_ladder[-1]
opaque_answer = {"AdaAccess": "allow"}
fixed_answer = inherit_properties(hard["graph"], hard["properties"], hard["defaults"], hard["query"])

print("opaque has explanation", opaque_answer.get("evidence") is not None)
print("fixed allow score", fixed_answer["scores"].get("allow"))
print("fixed evidence steps", len(fixed_answer["evidence"]))


## Evaluate it + Practice

- Metric: property correctness and explanation-step count, compared with a no-skill baseline that guesses or accepts the first candidate.
- Sanity check: rerun D1 and verify the hand-counted lesson numbers before trusting larger rungs.
- Ablation: turn off the key symbolic check and confirm the metric drops on D3-D5.
- Failure signals: unexplained answers, fewer checked cases than the rung size requires, or a contradiction accepted as valid.
- Keep all instances CPU-only, seeded, and small enough to inspect.

Practice: Add a new Mammal parent to D2 and trace the inherited property.

Practice: Change the D3 default scores so flying is no longer blocked.

Practice: Design an opaque D5 label and list the audit questions it cannot answer.